# Initial-margin CVaR MKABSDE

This notebook displays the retained quadrature campaign, both proxy BSDEs, and the common-path comparison of three conditional-CVaR estimators. Both payoffs use 256 full-law Gauss–Legendre nodes and Hermite degree two. The scheme solves for the MVA; future losses use the reconstructed total value.

## Complete reproduction commands

```bash
python -m initial_margin.run_campaign \
  --product all --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.compare_risk_estimators \
  --product all --steps 200 --seeds 42 43 44 \
  --paths 200000 --degree 2 --device cuda \
  --deltas 0.02 0.10 0.50 --output /tmp/absde-es-audit.json

python -m initial_margin.deterministic_reference \
  --product all --steps 50 100 200 --space-steps 6000 \
  --expectation-nodes 24 \
  --output initial_margin/results/deterministic_reference.json
```

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
sys.path.insert(0, str(root))
from initial_margin.evaluation import ARTICLE_REFERENCE_VALUES

result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')
from tools.mva_robustness import ESTIMATORS, method_comparison, spatial_comparison

## Relative margin valuation adjustment

Every number is $100\,\mathrm{MVA}_0/V_0^{bs}=100\,(V_0-V_0^{bs})/V_0^{bs}$, expressed in percent. The raw MVA and total price remain in the JSON artifacts; this is not the CVaR collateral.

In [2]:
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    base_price = payload['rows'][0]['metrics']['black_scholes_no_im']['price']
    article = ARTICLE_REFERENCE_VALUES[product]
    degree = payload['summary'][0]['hermite_degree']
    lines = [
        f'### ATM {product} — Hermite degree {degree}',
        '',
        '| $N$ | direct MKABSDE | nonlinear proxy | linear proxy |',
        '|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {row['direct_mkabsde_mva_percent_of_black_scholes_mean']:.6f} ± {row['direct_mkabsde_mva_percent_of_black_scholes_std']:.2e} | "
            f"{row['nonlinear_proxy_mva_percent_of_black_scholes_mean']:.6f} ± {row['nonlinear_proxy_mva_percent_of_black_scholes_std']:.2e} | "
            f"{row['linear_proxy_mva_percent_of_black_scholes_mean']:.6f} ± {row['linear_proxy_mva_percent_of_black_scholes_std']:.2e} |"
        )
    lines.append(
        f"| Reference value from Agarwal et al. | — | ≈ {100 * (article['nonlinear_proxy_price'] - base_price) / base_price:.2f} | ≈ {100 * (article['linear_proxy_price'] - base_price) / base_price:.2f} |"
    )
    display(Markdown('\n'.join(lines)))

### ATM call — Hermite degree 2

| $N$ | direct MKABSDE | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|
| 50 | 0.861560 ± 1.31e-03 | 1.010607 ± 1.32e-03 | 1.007173 ± 1.32e-03 |
| 100 | 0.859312 ± 9.07e-04 | 1.007756 ± 9.32e-04 | 1.004309 ± 9.30e-04 |
| 200 | 0.858423 ± 1.10e-03 | 1.006538 ± 1.14e-03 | 1.003083 ± 1.14e-03 |
| Reference value from Agarwal et al. | — | ≈ 1.00 | ≈ 1.06 |

### ATM butterfly — Hermite degree 2

| $N$ | direct MKABSDE | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|
| 50 | 0.765830 ± 7.52e-04 | 0.724722 ± 6.56e-04 | 0.723675 ± 6.56e-04 |
| 100 | 0.773317 ± 6.21e-04 | 0.732213 ± 5.26e-04 | 0.731134 ± 5.27e-04 |
| 200 | 0.775954 ± 4.33e-04 | 0.735188 ± 2.07e-04 | 0.734097 ± 2.07e-04 |
| Reference value from Agarwal et al. | — | ≈ 0.70 | ≈ 0.70 |

## Larger margin periods

Only the margin horizon changes: Delta = 0.02, 0.10, 0.50 years. All runs use Hermite degree two, N = 200, 200,000 paths and three independent seeds. Entries are mean ± sample SD of 100 MVA / V_BS (%); the SD is in percentage points, not a numerical-error bound. These are stress tests of the small-Delta proxies, not usual financial margin periods.

    python -m initial_margin.margin_periods --device cuda --output-dir /tmp/absde-margin-table

Use a fresh output directory. The README in initial_margin lists the independent space/time/quadrature refinement commands. This notebook reads the retained JSON only.

In [3]:
horizons = json.loads((result_root / 'margin_periods' / 'summary.json').read_text())
lines = ['| Product | Delta (years) | Direct MKABSDE | Nonlinear proxy | Linear proxy |',
         '|---|---:|---:|---:|---:|']
for row in horizons:
    values = [f"{row['methods'][name]['mva_percent_mean']:.4f} ± {row['methods'][name]['mva_percent_std']:.4f}"
              for name in ('direct', 'nonlinear', 'linear')]
    lines.append(f"| {row['product']} | {row['margin_period']:.2f} | " + ' | '.join(values) + ' |')
display(Markdown('\n'.join(lines)))

| Product | Delta (years) | Direct MKABSDE | Nonlinear proxy | Linear proxy |
|---|---:|---:|---:|---:|
| call | 0.02 | 0.8586 ± 0.0015 | 1.0066 ± 0.0015 | 1.0032 ± 0.0015 |
| call | 0.10 | 1.5177 ± 0.0030 | 2.2000 ± 0.0033 | 2.1836 ± 0.0033 |
| call | 0.50 | 1.8903 ± 0.0042 | 4.2725 ± 0.0059 | 4.2114 ± 0.0059 |
| butterfly | 0.02 | 0.7756 ± 0.0003 | 0.7352 ± 0.0003 | 0.7341 ± 0.0003 |
| butterfly | 0.10 | 1.4921 ± 0.0008 | 1.5059 ± 0.0006 | 1.5016 ± 0.0006 |
| butterfly | 0.50 | 2.0144 ± 0.0010 | 2.4681 ± 0.0008 | 2.4592 ± 0.0008 |

At Delta = 0.50, the direct MVA is about 56% below the nonlinear proxy for the call and 18.4% below for the butterfly. For the butterfly the gap changes sign: it was about 5.5% above at Delta = 0.02. Refinement preserves these large-horizon gaps, but neither the references nor the across-seed dispersion prove a uniform-in-Delta numerical error bound.

## Robustness: conditional ES and spatial regression

All rows report MVA as a percentage of the Black–Scholes price. The direct solvers share the outer paths, continuation fit and pathwise MVA update. No control or extra Hermite fit of the risk is used. Nested MC fits the updated MVA on the spots with inner samples. Means and sample SDs use three matched seeds.

In [4]:
lines = ['| Product | Horizon | Quadrature | Nested MC | VaR/ES learning |',
         '|---|---:|---:|---:|---:|']
for row in method_comparison(result_root):
    cells = [f"{row['methods'][m]['mean']:.4f} ± {row['methods'][m]['sd']:.4f}" for m in ESTIMATORS]
    lines.append(f"| {row['product']} | {row['margin_period']:.2f} | " + " | ".join(cells) + " |")
display(Markdown("\n".join(lines)))

| Product | Horizon | Quadrature | Nested MC | VaR/ES learning |
|---|---:|---:|---:|---:|
| call | 0.02 | 0.8586 ± 0.0015 | 0.8594 ± 0.0012 | 0.8607 ± 0.0026 |
| call | 0.10 | 1.5177 ± 0.0030 | 1.5186 ± 0.0041 | 1.5235 ± 0.0054 |
| call | 0.50 | 1.8903 ± 0.0042 | 1.8910 ± 0.0073 | 1.9207 ± 0.0148 |
| butterfly | 0.02 | 0.7756 ± 0.0003 | 0.7778 ± 0.0078 | 0.7858 ± 0.0041 |
| butterfly | 0.10 | 1.4921 ± 0.0008 | 1.4955 ± 0.0145 | 1.5090 ± 0.0015 |
| butterfly | 0.50 | 2.0144 ± 0.0010 | 2.0173 ± 0.0174 | 2.0746 ± 0.0117 |

The spatial comparison keeps the time grid and conditional ES quadrature fixed. Hermite values are ten-seed means at 0.02 and three-seed means at 0.10 and 0.50. The fine spatial grid is an independent numerical check, not an exact continuous-time solution.

In [5]:
lines = ['| Horizon | N | Call: Hermite | Call: fine grid | Butterfly: Hermite | Butterfly: fine grid |',
         '|---:|---:|---:|---:|---:|---:|']
for row in spatial_comparison(result_root):
    cells = [f"{row['products'][p][method]:.6f}" for p in products for method in ('hermite', 'fine_grid')]
    lines.append(f"| {row['margin_period']:.2f} | {row['time_steps']} | " + " | ".join(cells) + " |")
display(Markdown("\n".join(lines)))

| Horizon | N | Call: Hermite | Call: fine grid | Butterfly: Hermite | Butterfly: fine grid |
|---:|---:|---:|---:|---:|---:|
| 0.02 | 50 | 0.861560 | 0.861569 | 0.765830 | 0.766698 |
| 0.02 | 100 | 0.859312 | 0.859493 | 0.773317 | 0.773731 |
| 0.02 | 200 | 0.858423 | 0.858241 | 0.775954 | 0.776302 |
| 0.10 | 200 | 1.517711 | 1.516532 | 1.492103 | 1.492860 |
| 0.50 | 200 | 1.890272 | 1.888571 | 2.014389 | 2.009895 |